# Стекинг, часть 1 · h1 и h8

Включите NVIDIA GPU и подключите исходные train/test parquet. В GPU-окружении нужны CuPy, cuML, XGBoost, LightGBM, CatBoost, sklearn и PyArrow. Для сравнения метамоделей нужны также SciPy и PyTorch. Запустите Save & Run All. Для всех четырёх частей выберите одинаковый PROFILE: full включает три режима train, fast — только пять стартов. Маршрутные Ridge используют фиксированный ROUTE_ALPHA=100 на стандартизованных признаках. Подбора alpha нет. SESSION_HOURS=11 ограничивает время вычислений одной сессии. При остановке сохраните output и подключите все ранее завершённые части с тем же профилем. При расширении fast до full подключите fast-выходы: завершённые five_starts будут использованы повторно. Результаты сохраняются в каталоге stack с идентификатором входов и кода. Для продолжения подключите сохранённые выходы целиком как Kaggle inputs. Если нужен короткий первый запуск, задайте `OOF_WEEKS=(1,)`, сохраните результат, подключите его в следующей сессии и верните все четыре недели.

1. Во всех четырёх частях выполните `MODE='oof'`, `ROLE='worker'`.
2. В части 1 подключите четыре результата, выберите `MODE='assemble'` и выполните Save & Run All. Получите OOF, результаты сравнения и winner.json. Для победителя среди исходных ансамблей сравниваются поправки 50% и 100% по каждой базовой колонке и по итоговому прогнозу. При проверке очередной недели множители используют только прошлые отложенные прогнозы.
3. Во всех четырёх частях подключите победителя и свои OOF-выходы, выполните `MODE='evaluate'`, `ROLE='worker'`. В части 1 подключите все четыре результата оценки и вывод assemble, выполните `MODE='evaluate'`, `ROLE='collect'`.
4. Во всех четырёх частях подключите отчёт оценки, победителя и свои OOF-выходы, выполните `MODE='submit'`, `ROLE='worker'`. В части 1 подключите четыре результата этого этапа и выходы assemble и evaluate/collect, установите `MODE='submit'`, `ROLE='collect'`. В каталоге stack появится submission.csv для full или submission_fast.csv для fast.

Четверть train определяется по датам текущего фолда. Все оценки используют старт 10:30. Независимая оценка test сохраняется до финального переобучения. Время и метрика конкурса выводятся по ходу запуска.

In [ ]:
from time import monotonic
SESSION_START = monotonic()
SESSION_HOURS = 11.0
PROFILE = 'full'
ROUTE_ALPHA = 100.0
ENGINE = []
PART = 1
HORIZONS = (1, 8)
MODE = 'oof'
ROLE = 'worker'
OOF_WEEKS = (1, 2, 3, 4)
INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
CODE_ID = 'e4a7d45d1c07daf82575e49b6ff8334a7763f412ca4f2c667d4c006daf630e99'
PARAMS = {'catboost_direct': {'1': {'algorithm': 'catboost',
                           'fold_scores': [0.2649253293536965,
                                           0.3393712503001195,
                                           0.2577104634869908],
                           'horizon': 1,
                           'params': {'bagging_temperature': 0.8803049874792026,
                                      'depth': 8,
                                      'l2_leaf_reg': 1.5679933916723008,
                                      'learning_rate': 0.03694755509074355,
                                      'random_strength': 2.052699079536471},
                           'residual': False,
                           'selection_end_exclusive': '2025-09-20 00:00:00',
                           'selection_labels_through': '2025-09-19 11:00:00',
                           'trees': 1997,
                           'trial': 7,
                           'tuning_score': 0.2873356810469356},
                     '8': {'algorithm': 'catboost',
                           'fold_scores': [0.3889594810488456,
                                           0.37977272327651856,
                                           0.3447420103337708],
                           'horizon': 8,
                           'params': {'bagging_temperature': 0.5824582803960838,
                                      'depth': 4,
                                      'l2_leaf_reg': 11.207606211860567,
                                      'learning_rate': 0.03691990689655602,
                                      'random_strength': 1.2958350559263474},
                           'residual': False,
                           'selection_end_exclusive': '2025-09-20 00:00:00',
                           'selection_labels_through': '2025-09-19 14:30:00',
                           'trees': 1003,
                           'trial': 3,
                           'tuning_score': 0.371158071553045}},
 'catboost_residual': {'1': {'algorithm': 'catboost',
                             'fold_scores': [0.29076623453160416,
                                             0.35361122609491846,
                                             0.2712612015871664],
                             'horizon': 1,
                             'params': {'bagging_temperature': 1.416145155592091,
                                        'depth': 4,
                                        'l2_leaf_reg': 53.99484409787433,
                                        'learning_rate': 0.0224831270535195,
                                        'random_strength': 1.8033450352296265},
                             'residual': True,
                             'selection_end_exclusive': '2025-09-20 00:00:00',
                             'selection_labels_through': '2025-09-19 11:00:00',
                             'trees': 2991,
                             'trial': 1,
                             'tuning_score': 0.30521288740456304},
                       '8': {'algorithm': 'catboost',
                             'fold_scores': [0.3958759915916222,
                                             0.3727780130275801,
                                             0.3485994991954041],
                             'horizon': 8,
                             'params': {'bagging_temperature': 0.5175599632000338,
                                        'depth': 4,
                                        'l2_leaf_reg': 1.1715937392307056,
                                        'learning_rate': 0.0542425537773747,
                                        'random_strength': 2.7279612062363463},
                             'residual': True,
                             'selection_end_exclusive': '2025-09-20 00:00:00',
                             'selection_labels_through': '2025-09-19 14:30:00',
                             'trees': 715,
                             'trial': 8,
                             'tuning_score': 0.37241783460486877}},
 'light_gbm_direct': {'1': {'algorithm': 'lightgbm',
                            'fold_scores': [0.25561795364171574,
                                            0.34312291570546305,
                                            0.2657814910425617],
                            'horizon': 1,
                            'params': {'colsample_bytree': 0.5325257964926398,
                                       'learning_rate': 0.021962256150304792,
                                       'min_child_samples': 80,
                                       'num_leaves': 53,
                                       'reg_alpha': 4.9352962094020985,
                                       'reg_lambda': 78.86714129990483,
                                       'subsample': 0.7096834432905521},
                            'residual': False,
                            'selection_end_exclusive': '2025-09-20 00:00:00',
                            'selection_labels_through': '2025-09-19 11:00:00',
                            'trees': 918,
                            'trial': 4,
                            'tuning_score': 0.2881741201299135},
                      '8': {'algorithm': 'lightgbm',
                            'fold_scores': [0.42386424437525133,
                                            0.40801187661167576,
                                            0.378471278322666],
                            'horizon': 8,
                            'params': {'colsample_bytree': 0.5325257964926398,
                                       'learning_rate': 0.021962256150304792,
                                       'min_child_samples': 80,
                                       'num_leaves': 53,
                                       'reg_alpha': 4.9352962094020985,
                                       'reg_lambda': 78.86714129990483,
                                       'subsample': 0.7096834432905521},
                            'residual': False,
                            'selection_end_exclusive': '2025-09-20 00:00:00',
                            'selection_labels_through': '2025-09-19 14:30:00',
                            'trees': 240,
                            'trial': 4,
                            'tuning_score': 0.4034491331031977}},
 'light_gbm_residual': {'1': {'algorithm': 'lightgbm',
                              'fold_scores': [0.2888701819950047,
                                              0.3607287755297878,
                                              0.27641407404877294],
                              'horizon': 1,
                              'params': {'colsample_bytree': 0.8925879806965068,
                                         'learning_rate': 0.03603084612270474,
                                         'min_child_samples': 46,
                                         'num_leaves': 20,
                                         'reg_alpha': 0.00015777663630582464,
                                         'reg_lambda': 3.4890188454913864,
                                         'subsample': 0.8096244944759625},
                              'residual': True,
                              'selection_end_exclusive': '2025-09-20 00:00:00',
                              'selection_labels_through': '2025-09-19 11:00:00',
                              'trees': 1010,
                              'trial': 3,
                              'tuning_score': 0.3086710105245218},
                        '8': {'algorithm': 'lightgbm',
                              'fold_scores': [0.4010769288530288,
                                              0.3885905891976863,
                                              0.3622007705187404],
                              'horizon': 8,
                              'params': {'colsample_bytree': 0.5780093202212182,
                                         'learning_rate': 0.1358198535217987,
                                         'min_child_samples': 107,
                                         'num_leaves': 33,
                                         'reg_alpha': 8.629132190071849e-05,
                                         'reg_lambda': 0.14936568554617632,
                                         'subsample': 0.8595304694689628},
                              'residual': True,
                              'selection_end_exclusive': '2025-09-20 00:00:00',
                              'selection_labels_through': '2025-09-19 14:30:00',
                              'trees': 8,
                              'trial': 0,
                              'tuning_score': 0.3839560961898185}},
 'rf_direct': {'1': {'algorithm': 'random_forest',
                     'fold_scores': [0.26783833607079893, 0.35015051786120815, 0.27551559034755535],
                     'horizon': 1,
                     'params': {'max_depth': 10,
                                'max_features': 0.6,
                                'max_samples': 0.8,
                                'min_samples_leaf': 32,
                                'n_estimators': 150},
                     'residual': False,
                     'selection_end_exclusive': '2025-09-20 00:00:00',
                     'selection_labels_through': '2025-09-19 11:00:00',
                     'trees': 150,
                     'trial': 0,
                     'tuning_score': 0.2978348147598541},
               '8': {'algorithm': 'random_forest',
                     'fold_scores': [0.37831892703498315, 0.3710947695256506, 0.3499387871323048],
                     'horizon': 8,
                     'params': {'max_depth': 12,
                                'max_features': 0.7252813963310067,
                                'max_samples': 0.5852620618436457,
                                'min_samples_leaf': 18,
                                'n_estimators': 300},
                     'residual': False,
                     'selection_end_exclusive': '2025-09-20 00:00:00',
                     'selection_labels_through': '2025-09-19 14:30:00',
                     'trees': 300,
                     'trial': 4,
                     'tuning_score': 0.36645082789764616}},
 'rf_residual': {'1': {'algorithm': 'random_forest',
                       'fold_scores': [0.32227336937702133,
                                       0.41038809352071964,
                                       0.28576469765737444],
                       'horizon': 1,
                       'params': {'max_depth': 12,
                                  'max_features': 0.7102471755674296,
                                  'max_samples': 0.5102922471479012,
                                  'min_samples_leaf': 84,
                                  'n_estimators': 222},
                       'residual': True,
                       'selection_end_exclusive': '2025-09-20 00:00:00',
                       'selection_labels_through': '2025-09-19 11:00:00',
                       'trees': 222,
                       'trial': 0,
                       'tuning_score': 0.33947538685170514},
                 '8': {'algorithm': 'random_forest',
                       'fold_scores': [0.39148559333574595, 0.38481879720446893, 0.336581951348399],
                       'horizon': 8,
                       'params': {'max_depth': 10,
                                  'max_features': 0.3608406803967395,
                                  'max_samples': 0.5325257964926398,
                                  'min_samples_leaf': 85,
                                  'n_estimators': 500},
                       'residual': True,
                       'selection_end_exclusive': '2025-09-20 00:00:00',
                       'selection_labels_through': '2025-09-19 14:30:00',
                       'trees': 500,
                       'trial': 2,
                       'tuning_score': 0.3709621139628713}},
 'ridge_direct': {'1': {'algorithm': 'ridge',
                        'fold_scores': [0.30661679341934034,
                                        0.43216873289270474,
                                        0.2827662781034614],
                        'horizon': 1,
                        'params': {'alpha': 52337.83899605336},
                        'residual': False,
                        'selection_end_exclusive': '2025-09-20 00:00:00',
                        'selection_labels_through': '2025-09-19 11:00:00',
                        'trees': None,
                        'trial': 17,
                        'tuning_score': 0.3405172681385022},
                  '8': {'algorithm': 'ridge',
                        'fold_scores': [0.33576362407439186,
                                        0.3314127512748996,
                                        0.32611358420660613],
                        'horizon': 8,
                        'params': {'alpha': 62457.60287469883},
                        'residual': False,
                        'selection_end_exclusive': '2025-09-20 00:00:00',
                        'selection_labels_through': '2025-09-19 14:30:00',
                        'trees': None,
                        'trial': 7,
                        'tuning_score': 0.3310966531852992}},
 'ridge_residual': {'1': {'algorithm': 'ridge',
                          'fold_scores': [0.31396064355328385,
                                          0.42332816774137294,
                                          0.3082249427828566],
                          'horizon': 1,
                          'params': {'alpha': 68303.46849054066},
                          'residual': True,
                          'selection_end_exclusive': '2025-09-20 00:00:00',
                          'selection_labels_through': '2025-09-19 11:00:00',
                          'trees': None,
                          'trial': 13,
                          'tuning_score': 0.3485045846925045},
                    '8': {'algorithm': 'ridge',
                          'fold_scores': [0.3577104710570939,
                                          0.3718551959791049,
                                          0.331224979956808],
                          'horizon': 8,
                          'params': {'alpha': 665600.7511302697},
                          'residual': True,
                          'selection_end_exclusive': '2025-09-20 00:00:00',
                          'selection_labels_through': '2025-09-19 14:30:00',
                          'trees': None,
                          'trial': 12,
                          'tuning_score': 0.3535968823310022}},
 'xgb_direct': {'1': {'algorithm': 'xgb',
                      'fold_scores': [0.253426219214185, 0.33500379741646447, 0.27286198632448244],
                      'horizon': 1,
                      'params': {'colsample_bytree': 0.5325257964926398,
                                 'learning_rate': 0.021962256150304792,
                                 'max_depth': 7,
                                 'min_child_weight': 16.40928673064792,
                                 'reg_alpha': 4.9352962094020985,
                                 'reg_lambda': 78.86714129990483,
                                 'subsample': 0.7096834432905521},
                      'residual': False,
                      'selection_end_exclusive': '2025-09-20 00:00:00',
                      'selection_labels_through': '2025-09-19 11:00:00',
                      'trees': 1367,
                      'trial': 4,
                      'tuning_score': 0.2870973343183773},
                '8': {'algorithm': 'xgb',
                      'fold_scores': [0.425386196323662, 0.3981609457331616, 0.37374446360596025],
                      'horizon': 8,
                      'params': {'colsample_bytree': 0.6558555380447055,
                                 'learning_rate': 0.12495137958834718,
                                 'max_depth': 3,
                                 'min_child_weight': 3.2927591344236156,
                                 'reg_alpha': 0.013194961490425657,
                                 'reg_lambda': 4.366473592979633,
                                 'subsample': 0.8818827995238937},
                      'residual': False,
                      'selection_end_exclusive': '2025-09-20 00:00:00',
                      'selection_labels_through': '2025-09-19 14:30:00',
                      'trees': 76,
                      'trial': 6,
                      'tuning_score': 0.3990972018875946}},
 'xgb_residual': {'1': {'algorithm': 'xgb',
                        'fold_scores': [0.2907016879479988,
                                        0.36037935526845194,
                                        0.2822723845133465],
                        'horizon': 1,
                        'params': {'colsample_bytree': 0.7159725093210578,
                                   'learning_rate': 0.028941442788596318,
                                   'max_depth': 4,
                                   'min_child_weight': 4.059611610484305,
                                   'reg_alpha': 0.000558952420521792,
                                   'reg_lambda': 6.847920095574778,
                                   'subsample': 0.8336647510712832},
                        'residual': True,
                        'selection_end_exclusive': '2025-09-20 00:00:00',
                        'selection_labels_through': '2025-09-19 11:00:00',
                        'trees': 2730,
                        'trial': 2,
                        'tuning_score': 0.31111780924326576},
                  '8': {'algorithm': 'xgb',
                        'fold_scores': [0.40217794296162224,
                                        0.39051138969553406,
                                        0.3529250200406154],
                        'horizon': 8,
                        'params': {'colsample_bytree': 0.5325257964926398,
                                   'learning_rate': 0.021962256150304792,
                                   'max_depth': 7,
                                   'min_child_weight': 16.40928673064792,
                                   'reg_alpha': 4.9352962094020985,
                                   'reg_lambda': 78.86714129990483,
                                   'subsample': 0.7096834432905521},
                        'residual': True,
                        'selection_end_exclusive': '2025-09-20 00:00:00',
                        'selection_labels_through': '2025-09-19 14:30:00',
                        'trees': 96,
                        'trial': 4,
                        'tuning_score': 0.38187145089925717}}}

In [ ]:
ENGINE.append(r'''
from pathlib import Path
from tempfile import TemporaryDirectory
import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

START = pd.Timestamp('2025-09-20')
TEST_START = pd.Timestamp('2025-10-18')
DAY = pd.Timedelta(days=1)
STEP = pd.Timedelta(minutes=30)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
TRAIN_MINUTES = (570, 600, 630, 660, 690)
REGIMES = {'full':('five_starts','mixed_starts','recent_all'), 'fast':('five_starts',)}[PROFILE]
FAMILIES = ('rf', 'light_gbm', 'catboost', 'xgb', 'ridge')
MODEL_NAMES = [f'{f}_{v}' for f in FAMILIES for v in ('direct', 'residual')]
KEY = ['cutoff', 'route_id', 'horizon']

TARGET_SCALE = 100000
ETA_CONTEXT = None
THREADS = min(4, os.cpu_count() or 1)

def file_digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def locate(name):
    hits = list(INPUT.rglob(name))
    if len(hits) != 1:
        raise ValueError(f'Ожидается один {name}, найдено {hits}')
    return hits[0]

def initialize():
    global INPUT, OUTPUT, ROOT, TRAIN_PATH, TEST_PATH, RUN_ID, CHECKPOINTS
    INPUT = Path(INPUT_DIR)
    TRAIN_PATH = locate('train_solo_track.parquet')
    TEST_PATH = locate('test_solo_track.parquet')
    fingerprints = [file_digest(TRAIN_PATH),file_digest(TEST_PATH)]
    identity = [CODE_ID,str(ROUTE_ALPHA),*fingerprints]
    if not np.isfinite(ROUTE_ALPHA) or ROUTE_ALPHA<=0:
        raise ValueError('ROUTE_ALPHA должен быть положительным')
    RUN_ID = hashlib.sha256('|'.join(identity).encode()).hexdigest()[:24]
    OUTPUT = Path(OUTPUT_DIR)
    ROOT = OUTPUT / f'stack_{RUN_ID}'
    ROOT.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS = [p for p in INPUT.rglob(ROOT.name) if p.is_dir() and p.resolve() != ROOT.resolve()]
    global LEGACY_ROOTS, LEGACY_ID
    legacy_code = '3b8eeac831b45fb4d966222cd335cbd8766c6ed6b2d243b63056fc4afea15b79'
    LEGACY_ID = hashlib.sha256('|'.join([legacy_code,*fingerprints]).encode()).hexdigest()[:24]
    LEGACY_ROOTS = [p for p in INPUT.rglob(f'stack_{LEGACY_ID}') if p.is_dir()]
    print(f'Конфигурация {RUN_ID}, часть {PART}, режим {MODE}, роль {ROLE}')

def profile_path(rel):
    name = str(rel)
    if PROFILE=='fast':
        if name.startswith('meta/'):
            return name.replace('meta/','meta_fast/',1)
        if '/timings/' in name:
            return name.replace('/timings/','/timings_fast/',1)
    return name

def existing(rel):
    rel = profile_path(rel)
    dest = ROOT / rel
    if dest.is_file():
        return dest
    hits = [base / rel for base in CHECKPOINTS if (base / rel).is_file()]
    if not hits:
        return None
    if len(hits) > 1 and len({file_digest(p) for p in hits}) != 1:
        raise ValueError(f'Несогласованные копии {rel}: {hits}')
    dest.parent.mkdir(parents=True, exist_ok=True)
    temp = dest.with_suffix('.copying')
    shutil.copyfile(hits[0], temp)
    temp.replace(dest)
    return dest

def require(rel):
    path = existing(rel)
    if path is None:
        raise FileNotFoundError(f'Подключите результат {rel} из запуска {RUN_ID}')
    return path

def save(rel, value):
    path = ROOT / profile_path(rel)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.partial')
    if isinstance(value, pd.DataFrame):
        value.to_parquet(temp, index=False)
    else:
        temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)

def load_json(rel):
    return json.loads(require(rel).read_text())

def train_indices(h, boundary, regime):
    possible = np.flatnonzero(times[:len(times)-h] + h * STEP < boundary)
    days = times[possible].normalize()
    unique = days.unique()
    early = days.isin(unique[:len(unique)//4])
    five = np.isin(times[possible].hour * 60 + times[possible].minute, TRAIN_MINUTES)
    masks = {'five_starts': five, 'mixed_starts': ~early | five, 'recent_all': ~early}
    return possible[masks[regime]]

def query_indices(boundary, days):
    if days == 0:
        return np.array([len(times)-1])
    grid = pd.date_range(boundary + pd.Timedelta(hours=10, minutes=30), periods=days, freq='D')
    result = times.get_indexer(grid)
    if (result < 0).any():
        raise ValueError(f'Нет стартов для {grid}')
    return result

def metric(y, p):
    y = np.asarray(y, dtype=np.float64)
    error = np.asarray(p, dtype=np.float64) - y
    if not len(y) or not np.isfinite(error).all():
        raise ValueError('Пустая выборка или нечисловой прогноз')
    volume = float(y.sum())
    return totals(len(y), volume, float(np.abs(error).sum()), float(error.sum()))

def totals(n, volume, absolute, signed):
    score = (absolute+abs(signed))/volume if volume else None
    return dict(rows=int(n), volume=volume, absolute=absolute, signed=signed,
                WAPE=absolute/volume if volume else None, relative_bias=signed/volume if volume else None,
                score=score, **{'WAPE + |Relative Bias|':score})

def combine(rows):
    return totals(*(sum(row[k] for row in rows) for k in ('rows','volume','absolute','signed')))

def task_path(h, stage, regime, name):
    return f'h{h}/{stage}/{regime}/{name}.parquet'

def complete(rel):
    if existing(rel + '.json') is not None and existing(rel) is not None:
        return True
    parts = Path(rel).parts
    if len(parts)!=4 or not parts[1].startswith('oof') or Path(rel).stem not in [*MODEL_NAMES,'profile28']:
        return False
    for root in globals().get('LEGACY_ROOTS',[]):
        data = root/rel
        manifest = root/(rel+'.json')
        if not (data.is_file() and manifest.is_file()):
            continue
        info = json.loads(manifest.read_text())
        boundary,_ = stage_spec(parts[1])
        if info['run_id']!=LEGACY_ID or pd.Timestamp(info['boundary'])!=boundary or not pd.Timestamp(info['fit_through'])<boundary:
            raise ValueError(f'Некорректный исходный OOF {data}')
        dest = ROOT/rel
        dest.parent.mkdir(parents=True,exist_ok=True)
        tmp = dest.with_suffix('.copying')
        shutil.copyfile(data,tmp)
        tmp.replace(dest)
        save(rel+'.json',dict(info,run_id=RUN_ID,source_run_id=LEGACY_ID))
        return True
    return False

def result_frame(h, origins):
    frame = pd.DataFrame(dict(cutoff=times[origins].repeat(len(routes)), route_id=np.tile(routes, len(origins))))
    frame['horizon'] = h
    frame['timestamp'] = frame.cutoff + h * STEP
    if origins[-1] + h < len(times):
        frame['y'] = target[origins+h].ravel()
    return frame

def finish_fit(rel, frame, pred, train_row, boundary, fit_through, seconds, device):
    if not np.isfinite(pred).all() or np.min(pred) < 0:
        raise ValueError(f'Некорректный прогноз {rel}')
    rows = {'train':train_row}
    if 'y' in frame:
        rows['test' if '/test/' in rel else 'held_out'] = metric(frame.y, pred)
    save(rel, frame.assign(prediction=pred))
    save(rel+'.json',dict(run_id=RUN_ID, boundary=str(boundary), fit_through=str(fit_through),
                          seconds=seconds, backend=device, metrics=rows))
    display(pd.DataFrame(rows).T.assign(task=rel))
    show_eta()

def show_eta():
    if not ETA_CONTEXT:
        return
    known = ETA_CONTEXT['completed']
    oof = [v for (h,s),v in known.items() if s.startswith('oof')]
    if not oof:
        return
    current = ETA_CONTEXT['current']
    remaining = 0.
    for h,stage in ETA_CONTEXT['pending']:
        if (h,stage) in known:
            continue
        similar = [v for (other,s),v in known.items() if s==stage]
        if not similar:
            similar = [v for (other,s),v in known.items() if s[:3]==stage[:3]]
        estimate = np.mean(similar or oof)
        if (h,stage)==current:
            estimate = max(0.,estimate-(time.monotonic()-ETA_CONTEXT['start']))
        remaining += estimate
    remaining += ETA_CONTEXT['future_features']*ETA_CONTEXT['features']
    print(f'ETA части по измеренным этапам: {remaining/3600:.2f} ч')

def stage_spec(stage):
    if stage.startswith('oof'):
        return START + (int(stage[-1])-1)*7*DAY, 7
    if stage == 'test':
        return TEST_START, 14
    if stage == 'extra':
        return TEST_START + 7*DAY, 7
    if stage == 'full':
        return times[-1] + STEP, 0
    raise ValueError(stage)

def device_setup():
    global xp, cp, cupyx, BACKEND, LGB_DEVICE
    try:
        import cupy as cp
        import cupyx
        from cuml.ensemble import RandomForestRegressor
        device_name = cp.cuda.runtime.getDeviceProperties(0)['name']
    except (ImportError,RuntimeError) as error:
        raise RuntimeError('Для расчёта баз включите NVIDIA GPU и подключите CuPy и cuML, совместимые с CUDA сессии.') from error
    xp, BACKEND = cp, 'cuda'
    cp.linalg.solve(cp.eye(2),cp.ones(2))
    LGB_DEVICE = 'cpu'
    for candidate in ['cuda','gpu']:
        code = f"""import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
r = np.tile(np.arange({len(routes)}),32)
rng = np.random.default_rng(42)
data = pd.DataFrame(dict(route_id=pd.Categorical(r),volume=rng.random(len(r))))
model = LGBMRegressor(device_type='{candidate}',n_estimators=2,max_cat_to_onehot=1,verbosity=-1,n_jobs={THREADS})
model.fit(data,r%17+data.volume,categorical_feature=['route_id'])
"""
        probe = subprocess.run([sys.executable,'-c',code],capture_output=True,text=True)
        if probe.returncode == 0:
            LGB_DEVICE = candidate
            break
        print(candidate, (probe.stderr or probe.stdout)[-1200:])
    print('GPU:',device_name,'LightGBM:',LGB_DEVICE)
    from importlib.metadata import version
    save(f'part{PART}/{MODE}_devices.json',dict(gpu=str(device_name),lightgbm_device=LGB_DEVICE,
         packages={k:version(k) for k in ('numpy','pandas','xgboost','lightgbm','catboost','scikit-learn')},
         cupy=cp.__version__,cuml=__import__('cuml').__version__))
''')

In [ ]:
ENGINE.append(r'''
initialize()
''')

In [ ]:
ENGINE.append(r'''
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
''')

In [ ]:
ENGINE.append(r'''
target = values[TARGET]
if MODE != 'assemble' and ROLE != 'collect':
    target = values[TARGET]

    def past(array, positions):
        positions = np.asarray(positions, dtype=np.int64)
        valid = (positions >= 0) & (positions < len(array))
        result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
        result[~valid] = np.nan
        return result

    def average_available(matrices):
        total = np.zeros_like(matrices[0], dtype=np.float32)
        count = np.zeros_like(total)
        for matrix in matrices:
            valid = np.isfinite(matrix)
            total += np.where(valid, matrix, 0)
            count += valid
        return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

    def rolling(array, steps):
        return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
    means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
    level24 = rolling(target, 48)
    positions = np.arange(len(times))
    extra_signals = {}
    for name in STATUSES[:3]:
        extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
    extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
    extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
    for name in STATUSES[3:]:
        extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
    extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
    extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
    extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
    hub_signal = extra_signals['status_6_diff_hourly']
    extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
    recent = means[TARGET, 4]
    usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
    extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

    def offset_name(base, offset):
        if offset == 0:
            return base
        return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

    def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
        for offset in offsets:
            common[offset_name(base, offset)] = past(array, anchor + offset)

    def common_features(origins):
        common = {}
        local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
        for lag in local_target_lags:
            common[f'target_back_{lag}'] = past(target, origins - lag)
        causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
        for name, array in values.items():
            for steps in (4, 12):
                add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
            if name in STATUSES:
                common[f'{name}_latest'] = array[origins]
                for lag in range(1, 5):
                    common[f'{name}_back_{lag}'] = past(array, origins - lag)
                if name in STATUSES[:3]:
                    change = array - past(array, positions - 2)
                    add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
        for name, array in extra_signals.items():
            add_window(common, name, array, origins, causal_offsets)
        for lag in (48, 96):
            add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
        add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
        add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
        return common
    CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

    for name, series in values.items():
        network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
        for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
            extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)


    def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
        grid = pd.DatetimeIndex(timestamps)
        route_ids = np.asarray(route_ids)
        target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
        if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
            raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
        if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
            raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
        if len(np.unique(route_ids)) != len(route_ids):
            raise ValueError('Идентификаторы маршрутов повторяются')
        steps = float(max_lookback_hours) * 2
        if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
            raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
        if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
            raise ValueError('min_pairs должен быть целым числом >=3')
        if not 0 <= min_correlation <= 1 or route_block_size < 1:
            raise ValueError('Некорректные параметры выбора корреляции')
        max_lag = int(steps)
        stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
        anchors = np.arange(max_lag + 2, stop)
        best_corr = np.full(len(route_ids), np.nan)
        best_lag = np.full(len(route_ids), np.nan)
        best_pairs = np.zeros(len(route_ids), dtype=np.int32)
        for start in range(0, len(route_ids), route_block_size):
            finish = min(start + route_block_size, len(route_ids))
            x = target_values[anchors, start:finish].astype(np.float64)
            for lag in range(2, max_lag + 1):
                z = status6_diff[anchors - lag, start:finish].astype(np.float64)
                valid = np.isfinite(x) & np.isfinite(z)
                count = valid.sum(axis=0)
                xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
                divisor = np.maximum(count, 1)
                sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
                covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
                vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
                vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
                denominator = np.sqrt(vx * vz)
                corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
                corr = np.clip(corr, -1, 1)
                previous = best_corr[start:finish]
                better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
                best_corr[start:finish][better] = corr[better]
                best_lag[start:finish][better] = lag
                best_pairs[start:finish][better] = count[better]
        accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
        return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

    def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
        if h not in range(1, 9) or chunk_rows < 1:
            raise ValueError('Некорректный горизонт или размер блока')
        extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
        names = CORRELATION_COLUMNS + extra_columns
        result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
        if lag_table is None:
            names = [prefix + name for name in names]
            return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
        by_route = lag_table.set_index('route_id').reindex(routes)
        route_index = pd.Index(routes)
        lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
        for start in range(0, len(frame), chunk_rows):
            finish = min(start + chunk_rows, len(frame))
            block = frame.iloc[start:finish]
            origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
            route = route_index.get_indexer(block.route_id)
            if (origin < 0).any() or (route < 0).any():
                raise ValueError('Строка не сопоставлена с историей')
            selected = lag[route]
            for j, name in enumerate(CORRELATION_COLUMNS[:4]):
                result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
            for offset in WINDOW_OFFSETS:
                source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
                available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
                signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
                available &= np.isfinite(signal)
                result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
                result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
        names = [prefix + name for name in names]
        frame = frame.drop(columns=names, errors='ignore')
        return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    def remove_seasonality(series):
        typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
        return series - typical

    residual_target = remove_seasonality(target)
    residual_signal = remove_seasonality(hub_signal)

    def fit_lags(boundary):
        tables = []
        for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
            table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
            tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
        return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

    def add_correlation_features(frame,h,lag_table):
        frame = correlation_block(frame,h,lag_table,hub_signal,'')
        if lag_table is None:
            return correlation_block(frame,h,None,residual_signal,'residual_')
        residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
        residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
        return correlation_block(frame,h,residual_table,residual_signal,'residual_')



    def horizon_dataset(h, origins):
        slot = origins + h
        dates = times[origins] + h * STEP
        common = common_features(origins)
        frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
        calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
        calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
        calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
        for name, array in calendar.items():
            frame[name] = np.repeat(array, len(routes))
        frame = add_correlation_features(frame, h, None)
        seasonal_columns = {}
        for offset in WINDOW_OFFSETS:
            for day in (1, 2, 7):
                seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
            for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
                seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
        for base in ('slot_daily7','slot_weekly4'):
            window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
            valid = np.isfinite(window)
            maximum = np.where(valid, window, -np.inf).max(axis=1)
            minimum = np.where(valid, window, np.inf).min(axis=1)
            peak = valid & (window == maximum[:,None])
            count = peak.sum(axis=1)
            shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
            seasonal_columns[base+'_peak_minutes'] = shift * 30
            seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
            seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
        frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
        profile = np.zeros((len(origins), len(routes)))
        counts = np.zeros_like(profile)
        for day in range(1, 29):
            level = past(level24, origins - day * 48)
            adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
            profile += np.nan_to_num(adjusted)
            counts += np.isfinite(adjusted)
        frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
        if h == 1:
            lower = np.zeros_like(target[origins])
            upper = past(target, origins - 2)
            for position in (origins - 1, origins):
                total = past(target, position)
                lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
            seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
            frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
        else:
            frame['overlap'] = np.nan
        raw_target_sources = []
        for lag in range(9):
            raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
        for lag in (48, 96):
            for offset in WINDOW_OFFSETS:
                raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
        for day in (1, 2, 7):
            for offset in WINDOW_OFFSETS:
                raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
        seen, redundant = (set(), [])
        for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
            if source in seen:
                redundant.append(name)
            else:
                seen.add(source)
        frame = frame.drop(columns=redundant)
        numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
        return frame.astype({name: 'float32' for name in numeric}).copy()
''')

In [ ]:
ENGINE.append(r'''
class HorizonData:
    def __init__(self, folder, h, stop=None):
        self.h = h
        self.stop = len(times) if stop is None else stop
        self.folder = Path(folder)
        self.corr_names = None
        self.corr = None
        self.frame = None
        self.baseline = np.empty((self.stop,len(routes)),np.float32)
        self.origins = np.arange(self.stop)
        if PROFILE=='fast':
            grid = times[:self.stop]
            self.origins = self.origins[np.isin(grid.hour*60+grid.minute,TRAIN_MINUTES)]
        self.slots = np.full(self.stop,-1,dtype=int)
        self.slots[self.origins] = np.arange(len(self.origins))
        for start in range(0,len(self.origins),32):
            idx = self.origins[start:start+32]
            frame = horizon_dataset(h,idx)
            if self.frame is None:
                self.cols = [c for c in frame if c not in ('cutoff','timestamp','route_id','profile28','overlap')
                             and not (c.startswith(('selected_status6','status6_lag_valid','lagged_status6','residual_')))]
                self.frame = np.memmap(self.folder/'static.bin', mode='w+', dtype='float32',
                                       shape=(len(self.origins),len(routes),len(self.cols)))
            self.frame[self.slots[idx]] = frame[self.cols].to_numpy().reshape(len(idx),len(routes),-1)
            self.baseline[idx] = frame.profile28.to_numpy().reshape(len(idx),len(routes))
        self.frame.flush()

    def prepare(self, boundary, stop):
        table = fit_lags(boundary)
        origins = self.origins[self.origins<stop]
        for start in range(0,len(origins),64):
            idx = origins[start:start+64]
            keys = result_frame(self.h,idx)[['cutoff','route_id']]
            corr = add_correlation_features(keys,self.h,table).drop(columns=['cutoff','route_id'])
            if self.corr is None:
                self.corr_names = corr.columns.tolist()
                self.corr = np.memmap(self.folder/'correlations.bin',mode='w+',dtype='float32',
                                      shape=(len(self.origins),len(routes),len(self.corr_names)))
            self.corr[self.slots[idx]] = corr.to_numpy().reshape(len(idx),len(routes),-1)

    def matrix(self, origins, route=None):
        slots = self.slots[origins]
        if (slots<0).any():
            raise ValueError('Запрошен старт вне выбранного профиля')
        if route is None:
            a,b = self.frame[slots], self.corr[slots]
        else:
            a,b = self.frame[slots,route], self.corr[slots,route]
        return np.concatenate([a,b],axis=-1).reshape(-1,len(self.cols)+len(self.corr_names))

def global_data(data, origins, label, only_finite):
    shape = (len(origins)*len(routes),len(data.cols)+len(data.corr_names))
    matrix = np.memmap(data.folder/f'{label}.bin',mode='w+',dtype='float32',shape=shape,order='F')
    count = 0
    keys = []
    for start in range(0,len(origins),32):
        chunk = origins[start:start+32]
        rows = result_frame(data.h,chunk)
        rows['profile28'] = data.baseline[chunk].ravel()
        keep = np.isfinite(rows.profile28.to_numpy())
        if not only_finite and not keep.all():
            raise ValueError('Неполный сезонный прогноз на оценке')
        batch = data.matrix(chunk)[keep]
        matrix[count:count+len(batch)] = batch
        count += len(batch)
        keys.append(rows.loc[keep])
    if count == 0:
        raise ValueError('Пустой train')
    return matrix[:count],pd.concat(keys,ignore_index=True)

def numeric_data(x, q, train_ids, query_ids, folder, scale):
    width = x.shape[1]
    bits = max(1,len(routes).bit_length())
    medians = np.empty(width)
    means = np.zeros(width)
    scales = np.ones(width)
    for col in range(width):
        values = x[:,col]
        valid = values[np.isfinite(values)]
        medians[col] = np.median(valid) if len(valid) else 0.
        if scale:
            filled = np.where(np.isfinite(values),values,medians[col])
            means[col] = np.mean(filled,dtype=np.float64)
            scales[col] = np.std(filled,dtype=np.float64) or 1.
    outputs = []
    for label,matrix,ids in [('train',x,train_ids),('query',q,query_ids)]:
        out = np.memmap(folder/f'encoded_{label}.bin',mode='w+',dtype='float32',
                        shape=(len(ids),width+bits))
        codes = pd.Index(routes).get_indexer(ids)+1
        for start in range(0,len(ids),16384):
            rows = slice(start,start+16384)
            block = matrix[rows]
            filled = np.where(np.isfinite(block),block,medians)
            out[rows,:width] = (filled-means)/scales
            for bit in range(bits):
                out[rows,width+bit] = (codes[rows] >> bit) & 1
        outputs.append(out)
    return outputs

def ridge_coefficients(x,y,alpha,backend):
    mean = np.mean(x,axis=0,dtype=np.float64)
    y_mean = np.mean(y,dtype=np.float64)
    gram = backend.zeros((x.shape[1],x.shape[1]),dtype=backend.float64)
    rhs = backend.zeros(x.shape[1],dtype=backend.float64)
    for start in range(0,len(x),16384):
        rows = slice(start,start+16384)
        a = backend.asarray(x[rows],dtype=backend.float64)-backend.asarray(mean)
        b = backend.asarray(y[rows]-y_mean,dtype=backend.float64)
        gram += a.T @ a
        rhs += a.T @ b
    gram += alpha*backend.eye(len(mean))
    coef = backend.linalg.solve(gram,rhs)
    coef = coef.get() if hasattr(coef,'get') else np.asarray(coef)
    return coef,float(y_mean - mean @ coef)

def linear_predict(x, coef, intercept):
    result = np.empty(len(x))
    for start in range(0,len(x),16384):
        result[start:start+16384] = x[start:start+16384] @ coef + intercept
    return result

def verify_global_ridge(x,y,alpha):
    rows = np.linspace(0,len(x)-1,min(1024,len(x)),dtype=int)
    a,b = np.asarray(x[rows],dtype=np.float64),y[rows]
    coef,offset = ridge_coefficients(a,b,alpha,xp)
    expected = Ridge(alpha=alpha,solver='lsqr',tol=1e-5).fit(a,b).predict(a)
    if not np.allclose(a @ coef+offset,expected,rtol=2e-4,atol=2e-4*max(1.,np.max(np.abs(b)))):
        raise ValueError('GPU Ridge расходится с sklearn')

def tree_frame(matrix, columns, ids, catboost):
    frame = pd.DataFrame(matrix,columns=columns,copy=False)
    frame.insert(0,'route_id',ids.astype(str).to_numpy() if catboost else pd.Categorical(ids,categories=routes))
    return frame

def fit_global(name, cfg, x,q,train,query, columns, folder, encoded=None):
    family = name.rsplit('_',1)[0]
    residual = name.endswith('_residual')
    baseline = train.profile28.to_numpy() if residual else 0.
    y = (train.y.to_numpy(dtype=np.float64)-baseline)/TARGET_SCALE
    params = cfg['params']
    if family in ('ridge','rf'):
        a,b = encoded if encoded is not None else numeric_data(x,q,train.route_id,query.route_id,folder,family=='ridge')
        if family == 'ridge':
            verify_global_ridge(a,y,params['alpha'])
            coef,intercept = ridge_coefficients(a,y,params['alpha'],xp)
            train_pred, pred = (linear_predict(z,coef,intercept) for z in (a,b))
        else:
            from cuml.ensemble import RandomForestRegressor
            model = RandomForestRegressor(random_state=42,n_streams=4,output_type='numpy',**params)
            model.fit(a,y.astype(np.float32))
            train_pred,pred = (np.concatenate([np.asarray(model.predict(z[i:i+32768])).reshape(-1)
                                               for i in range(0,len(z),32768)]) for z in (a,b))
        device = 'cuda'
    else:
        a,b = (tree_frame(z,columns,frame.route_id,family=='catboost') for z,frame in ((x,train),(q,query)))
        if family == 'xgb':
            from xgboost import XGBRegressor
            model = XGBRegressor(n_estimators=cfg['trees'],device='cuda',tree_method='hist',enable_categorical=True,
                                 max_cat_to_onehot=1,objective='reg:squarederror',random_state=42,n_jobs=THREADS,**params)
            model.fit(a,y)
            device = json.loads(model.get_booster().save_config())['learner']['generic_param']['device']
            if not device.startswith('cuda'):
                raise RuntimeError('XGBoost не использует GPU')
        elif family == 'catboost':
            from catboost import CatBoostRegressor
            model = CatBoostRegressor(iterations=cfg['trees'],loss_function='RMSE',task_type='GPU',random_seed=42,
                                      bootstrap_type='Bayesian',one_hot_max_size=1,allow_writing_files=False,
                                      thread_count=THREADS,verbose=False,**params)
            model.fit(a,y,cat_features=['route_id'])
            device = 'cuda'
        else:
            from lightgbm import LGBMRegressor
            model = LGBMRegressor(n_estimators=cfg['trees'],objective='regression',device_type=LGB_DEVICE,
                                  subsample_freq=1,max_cat_to_onehot=1,verbosity=-1,random_state=42,n_jobs=THREADS,**params)
            model.fit(a,y,categorical_feature=['route_id'])
            device = LGB_DEVICE
        train_pred,pred = (np.concatenate([model.predict(z.iloc[i:i+32768]) for i in range(0,len(z),32768)])
                           for z in (a,b))
    train_pred = np.maximum(0.,train_pred*TARGET_SCALE+baseline)
    pred = np.maximum(0.,pred*TARGET_SCALE+(query.profile28.to_numpy() if residual else 0.))
    return metric(train.y,train_pred),pred,device
''')

In [ ]:
ENGINE.append(r'''
from contextlib import nullcontext
def to_host(value):
    return cp.asnumpy(value) if BACKEND == 'cuda' else np.asarray(value)

def synchronize():
    if BACKEND == 'cuda':
        cp.cuda.get_current_stream().synchronize()

def linalg_guard():
    return cupyx.errstate(linalg='raise') if BACKEND == 'cuda' else nullcontext()

def totals_to_metrics(values):
    count, volume, predicted, absolute, signed = [float(value) for value in values]
    if count < 1 or not np.isfinite(values).all():
        raise ValueError('Нечисловые или пустые пакетные метрики')
    return dict(n=int(round(count)), actual_sum=volume, prediction_sum=predicted, absolute_error=absolute,
                signed_error=signed, WAPE=absolute / volume if volume > 0 else None,
                abs_relative_bias=abs(signed) / volume if volume > 0 else None,
                score=(absolute + abs(signed)) / volume if volume > 0 else None, MAE=absolute / count)

class DeviceBatch:
    def __init__(self, items, spectral):
        self.items = items
        self.size = len(items)
        self.width = max(item['x'].shape[1] for item in items)
        rows = max(len(item['y']) for item in items)
        query_rows = max(len(item['query_y']) for item in items)
        self.x = xp.zeros((self.size, rows, self.width), dtype=xp.float64)
        self.q = xp.zeros((self.size, query_rows, self.width), dtype=xp.float64)
        self.y = xp.zeros((self.size, rows), dtype=xp.float64)
        self.qy = xp.zeros((self.size, query_rows), dtype=xp.float64)
        self.mask = xp.zeros((self.size, rows), dtype=xp.bool_)
        self.original_mask = xp.zeros((self.size, rows), dtype=xp.bool_)
        self.query_mask = xp.zeros((self.size, query_rows), dtype=xp.bool_)
        self.y_mean = xp.asarray([item['y_mean'] for item in items], dtype=xp.float64)
        for index, item in enumerate(items):
            n, p = item['x'].shape
            qn = len(item['query_y'])
            self.x[index, :n, :p] = xp.asarray(item['x'])
            self.q[index, :qn, :p] = xp.asarray(item['query_x'])
            self.y[index, :n] = xp.asarray(item['y'])
            self.qy[index, :qn] = xp.asarray(item['query_y'])
            self.mask[index, :n] = True
            self.original_mask[index, :n] = xp.asarray(item['original_train'])
            self.query_mask[index, :qn] = True
        transposed = self.x.swapaxes(-1, -2)
        self.gram = transposed @ self.x
        centered_y = xp.where(self.mask, self.y - self.y_mean[:, None], 0.0)
        self.rhs = (transposed @ centered_y[:, :, None])[:, :, 0]
        if spectral:
            with linalg_guard():
                values, self.vectors = xp.linalg.eigh(self.gram)
            self.eigenvalues = xp.maximum(values, 0.0)
            self.projected = (self.vectors.swapaxes(-1, -2) @ self.rhs[:, :, None])[:, :, 0]
        synchronize()

    def coefficients(self, alphas, spectral):
        alpha = xp.asarray(alphas, dtype=xp.float64)
        if spectral:
            quotient = self.projected / (self.eigenvalues + alpha[:, None])
            result = (self.vectors @ quotient[:, :, None])[:, :, 0]
        else:
            system = self.gram.copy()
            index = xp.arange(self.width)
            system[:, index, index] += alpha[:, None]
            with linalg_guard():
                result = xp.linalg.solve(system, self.rhs[:, :, None])[:, :, 0]
        residual = (self.gram @ result[:, :, None])[:, :, 0] + alpha[:, None] * result - self.rhs
        relative = xp.sqrt(xp.sum(residual * residual, axis=1)) / xp.maximum(xp.sqrt(xp.sum(self.rhs * self.rhs, axis=1)), 1.0)
        if not bool(to_host(xp.isfinite(result).all())) or np.any(to_host(relative) > 1e-7):
            raise ValueError('Пакетное решение Ridge не прошло проверку невязки')
        return result

    def predictions(self, coefficients):
        train = xp.maximum((self.x @ coefficients[:, :, None])[:, :, 0] + self.y_mean[:, None], 0.0)
        query = xp.maximum((self.q @ coefficients[:, :, None])[:, :, 0] + self.y_mean[:, None], 0.0)
        return train, query

    def reduce(self, prediction, actual, mask):
        error = xp.where(mask, prediction - actual, 0.0)
        totals = xp.stack([mask.sum(axis=1), xp.where(mask, actual, 0.0).sum(axis=1),
                           xp.where(mask, prediction, 0.0).sum(axis=1), xp.abs(error).sum(axis=1), error.sum(axis=1)], axis=1)
        return [totals_to_metrics(values) for values in to_host(totals)]

    def metrics(self, coefficients, include_original=False):
        train, query = self.predictions(coefficients)
        result = dict(train=self.reduce(train, self.y, self.mask))
        if self.items[0]['phase'] != 'full':
            result['query'] = self.reduce(query, self.qy, self.query_mask)
        if include_original:
            result['original_train'] = self.reduce(train, self.y, self.original_mask)
        return result, to_host(query)
''')

In [ ]:
ENGINE.append(r'''
def route_features(data, origins, r):
    raw = data.matrix(origins,r)
    names = data.cols+data.corr_names
    keep = [i for i,c in enumerate(names) if c not in ('dayofweek','is_weekend')]
    hours = times[origins] + data.h*STEP
    angle = (hours.hour*2+hours.minute/30)*2*np.pi/48
    matrix = np.column_stack([raw[:,keep],np.sin(angle),np.cos(angle)]).astype(np.float64)
    matrix[~np.isfinite(matrix)] = np.nan
    return matrix

def route_item(data, origins, query, r, stage):
    a = route_features(data,origins,r)
    b = route_features(data,query,r)
    imputer = SimpleImputer(strategy='median',keep_empty_features=True)
    a = imputer.fit_transform(a)
    b = imputer.transform(b)
    active = np.flatnonzero(np.ptp(a,axis=0)>0)
    if not len(active):
        active = [0]
    scaler = StandardScaler()
    a = scaler.fit_transform(a[:,active])
    b = scaler.transform(b[:,active])
    mean = a.mean(axis=0)
    y = target[origins+data.h,r].astype(np.float64)
    qy = np.zeros(len(query)) if stage=='full' else target[query+data.h,r].astype(np.float64)
    return dict(x=a-mean,query_x=b-mean,y=y,y_mean=float(y.mean()),query_y=qy,
                original_train=np.ones(len(y),dtype=bool),phase=stage,r=r)

def route_result(items, reference):
    batch = DeviceBatch(items,spectral=False)
    chosen = np.full(len(items),ROUTE_ALPHA)
    coef = batch.coefficients(chosen,False)
    metrics, predictions = batch.metrics(coef)
    if reference:
        for row,item in enumerate(items[:2]):
            reference_model = Ridge(alpha=chosen[row],solver='cholesky').fit(item['x'],item['y'])
            expected = np.maximum(0,reference_model.predict(item['query_x']))
            if not np.allclose(expected,predictions[row,:len(expected)],rtol=1e-6,atol=max(1.,np.abs(item['y']).max())*1e-7):
                raise ValueError('Проверка маршрутной Ridge против sklearn не пройдена')
    return predictions,metrics

def route_with_retry(items,reference):
    try:
        return route_result(items,reference)
    except cp.cuda.memory.OutOfMemoryError:
        if len(items)==1:
            raise
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    mid = len(items)//2
    parts = [route_with_retry(items[s],reference)
             for s in (slice(0,mid),slice(mid,None))]
    return np.concatenate([v for v,_ in parts]),{k:parts[0][1][k]+parts[1][1][k] for k in parts[0][1]}

def route_prefix(h,stage,regime):
    return f'h{h}/routes/{stage}/{regime}'

def route_stage(data,stage,regime,origins,query,boundary):
    started = time.monotonic()
    prefix = route_prefix(data.h,stage,regime)
    outputs,train_rows = [],[]
    for start in range(0,len(routes),8):
        rel = f'{prefix}/{start}.json'
        if existing(rel) is None:
            batch_start = time.monotonic()
            ids = range(start,min(start+8,len(routes)))
            items = [route_item(data,origins,query,r,stage) for r in ids]
            out, metrics = route_with_retry(items,start==0)
            rows = {}
            for split,values in metrics.items():
                rows[split] = combine([totals(v['n'],v['actual_sum'],v['absolute_error'],v['signed_error']) for v in values])
            save(rel,dict(values=out.tolist(),metrics=rows,seconds=time.monotonic()-batch_start,backend='cuda',
                          alpha=ROUTE_ALPHA,run_id=RUN_ID,boundary=str(boundary),fit_through=str(times[origins[-1]]+data.h*STEP)))
            display(pd.DataFrame(rows).T.assign(batch=start,stage=stage,regime=regime))
        batch = load_json(rel)
        outputs.append(np.asarray(batch['values']))
        train_rows.append(batch['metrics']['train'])
    rel = task_path(data.h,stage,regime,'route_ridge')
    finish_fit(rel,result_frame(data.h,query),np.concatenate(outputs).T.ravel(),combine(train_rows),
                   boundary,times[origins[-1]]+data.h*STEP,time.monotonic()-started,'cuda')
''')

In [ ]:
ENGINE.append(r'''
def predictors(include_routes):
    families = [*MODEL_NAMES,*(['route_ridge'] if include_routes else [])]
    return [f'{r}__{name}' for r in REGIMES for name in families]+['profile28']

def jobs(h,stage,cols):
    return any(not complete(task_path(h,stage,*col.split('__'))) for col in cols if col!='profile28') or not complete(task_path(h,stage,'shared','profile28'))

def run_stage(data,stage,cols):
    h = data.h
    boundary,days = stage_spec(stage)
    query = query_indices(boundary,days)
    tic = time.monotonic()
    data.prepare(boundary,int(query[-1])+1)
    timing = dict(lag_seconds=time.monotonic()-tic,route_seconds=0.,matrix_seconds=0.,global_fit_seconds=0.)
    for regime in REGIMES:
        origins = train_indices(h,boundary,regime)
        if times[origins[-1]]+h*STEP >= boundary:
            raise ValueError(f'Утечка меток {stage}')
        route_rel = task_path(h,stage,regime,'route_ridge')
        if f'{regime}__route_ridge' in cols and not complete(route_rel):
            tic = time.monotonic()
            route_stage(data,stage,regime,origins,query,boundary)
            timing['route_seconds'] += time.monotonic()-tic
        names = [n for n in MODEL_NAMES if f'{regime}__{n}' in cols and not complete(task_path(h,stage,regime,n))]
        if not names:
            continue
        tic = time.monotonic()
        x, train = global_data(data,origins,'train',True)
        q, validation = global_data(data,query,'query',False)
        timing['matrix_seconds'] += time.monotonic()-tic
        encoded = None
        previous_family = None
        for name in names:
            family = name.rsplit('_',1)[0]
            if family!=previous_family:
                encoded = None
                if family in ('ridge','rf'):
                    tic = time.monotonic()
                    encoded = numeric_data(x,q,train.route_id,validation.route_id,data.folder,family=='ridge')
                    timing['matrix_seconds'] += time.monotonic()-tic
                previous_family = family
            started = time.monotonic()
            row,pred,device = fit_global(name,PARAMS[name][str(h)],x,q,train,validation,data.cols+data.corr_names,data.folder,encoded)
            finish_fit(task_path(h,stage,regime,name),result_frame(h,query),pred,row,boundary,
                       train.timestamp.max(),time.monotonic()-started,device)
            timing['global_fit_seconds'] += time.monotonic()-started
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()
        del x,train,q,validation,encoded
    rel = task_path(h,stage,'shared','profile28')
    if not complete(rel):
        origins = train_indices(h,boundary,'five_starts')
        p = data.baseline[origins].ravel()
        keep = np.isfinite(p)
        row = metric(target[origins+h].ravel()[keep],p[keep])
        finish_fit(rel,result_frame(h,query),data.baseline[query].ravel(),row,boundary,
                   times[origins[-1]]+h*STEP,0.,'seasonal')

    return timing

def run_worker():
    global ETA_CONTEXT
    if MODE=='oof':
        if not OOF_WEEKS or any(i not in range(1,5) for i in OOF_WEEKS):
            raise ValueError('OOF_WEEKS должен содержать номера недель 1–4')
        stages = [f'oof{i}' for i in sorted(set(OOF_WEEKS))]
        cols = predictors(True)
    else:
        winner = load_json('meta/winner.json')
        cols = winner['columns']
        stages = ['test'] if MODE=='evaluate' else ['extra','full']
        if MODE=='submit':
            if load_json('meta/test_report.json')['winner']!=winner:
                raise ValueError('Отчёт test относится к другому победителю')
    device_setup()
    schedule = [(h,s) for h in HORIZONS for s in stages if jobs(h,s,cols)]
    completed = {}
    for h in HORIZONS:
        for s in stages:
            rel = f'h{h}/timings/{s}.json'
            if (h,s) not in schedule and existing(rel):
                completed[h,s] = load_json(rel)['stage_seconds']
    ETA_CONTEXT = dict(pending=schedule,completed=completed)
    for h in HORIZONS:
        pending = [s for horizon,s in schedule if horizon==h]
        if not pending:
            continue
        with TemporaryDirectory() as tmp:
            start = time.monotonic()
            last_origin = max(query_indices(*stage_spec(s))[-1] for s in pending)
            data = HorizonData(tmp,h,int(last_origin)+1)
            feature_seconds = time.monotonic()-start
            print(f'h{h}: признаки {feature_seconds/60:.1f} мин')
            future = {other for other,_ in schedule if HORIZONS.index(other)>HORIZONS.index(h)}
            ETA_CONTEXT.update(future_features=len(future),features=feature_seconds)
            for stage in pending:
                start = time.monotonic()
                ETA_CONTEXT.update(current=(h,stage),start=start)
                timing = run_stage(data,stage,cols)
                duration = time.monotonic()-start
                completed[h,stage] = duration
                save(f'h{h}/timings/{stage}.json',dict(stage_seconds=duration,features_seconds=feature_seconds,**timing))
                display(pd.DataFrame([dict(horizon=h,stage=stage,seconds=duration,**timing)]))
                show_eta()
            del data
        gc.collect()
''')

In [ ]:
ENGINE.append(r'''
def collect_stage(stage,columns):
    boundary,days = stage_spec(stage)
    origins = query_indices(boundary,days)
    blocks = []
    for h in range(1,9):
        block = result_frame(h,origins).set_index(KEY)
        for col in columns:
            regime,name = ('shared','profile28') if col=='profile28' else col.split('__')
            rel = task_path(h,stage,regime,name)
            info = load_json(rel+'.json')
            if (info['run_id']!=RUN_ID or pd.Timestamp(info['boundary'])!=boundary
                    or not pd.Timestamp(info['fit_through'])<boundary):
                raise ValueError(f'Граница обучения {rel}')
            frame = pd.read_parquet(require(rel)).set_index(KEY)
            if not frame.index.is_unique or not block.index.difference(frame.index).empty or len(frame)!=len(block):
                raise ValueError(f'Неполные или повторяющиеся ключи {rel}')
            frame = frame.reindex(block.index)
            for key in ('timestamp','y'):
                if key in block and not np.array_equal(block[key],frame[key]):
                    raise ValueError(f'Несовпадение {key} в {rel}')
            pred = frame.prediction.to_numpy()
            if not np.isfinite(pred).all() or (pred<0).any():
                raise ValueError(f'Некорректные значения {rel}')
            block[col] = pred
        blocks.append(block.reset_index())
    return pd.concat(blocks,ignore_index=True).assign(stage=stage,fit_before=boundary)

def convex_weights(x,y):
    from scipy.optimize import linprog
    scale = y.sum()
    if scale<=0:
        raise ValueError('Нулевой объём meta-train')
    k = x.shape[1]
    weights = np.full(k,1/k)
    best,upper = weights.copy(),np.inf
    cuts = []
    for _ in range(2000):
        err = x @ weights-y
        value = (np.abs(err).sum()+abs(err.sum()))/scale
        if value<upper:
            best,upper = weights.copy(),value
        grad = (np.sign(err)+np.sign(err.sum())) @ x /scale
        cuts.append((np.r_[grad,-1.],grad @ weights-value))
        a,b = zip(*cuts)
        lp = linprog(np.r_[np.zeros(k),1.],A_ub=np.asarray(a),b_ub=b,A_eq=[np.r_[np.ones(k),0.]],
                     b_eq=[1.],bounds=[(0,1)]*k+[(0,None)],method='highs')
        if not lp.success:
            raise RuntimeError(lp.message)
        if upper-lp.fun<=1e-7:
            best = np.maximum(best,0.)
            return best/best.sum()
        weights = lp.x[:-1]
    raise RuntimeError('LP не достиг допуска 1e-7')

def meta_context(frame,columns):
    x = frame[columns].to_numpy(dtype=np.float64)
    return np.column_stack([x,x.std(axis=1),frame.horizon,frame.cutoff.dt.dayofweek])

def calibration_factors(actual, predicted, strength):
    predicted = np.asarray(predicted,dtype=float)
    ratio = np.divide(actual,predicted,out=np.ones_like(predicted),where=predicted>0)
    return (1-strength)+strength*ratio

def calibrate_bases(train,query,cols,strength):
    values = train[cols].to_numpy(dtype=float)
    calibrated = np.empty_like(values)
    weeks = (train.cutoff.dt.normalize()-START).dt.days//7
    volume = 0.
    predicted = np.zeros(len(cols))
    for week in sorted(weeks.unique()):
        mask = weeks.eq(week)
        raw = values[mask]
        calibrated[mask] = raw*calibration_factors(volume,predicted,strength)
        volume += train.loc[mask,'y'].sum()
        predicted += raw.sum(axis=0)
    factors = calibration_factors(volume,predicted,strength)
    future = query[cols].to_numpy(dtype=float)*factors
    return train.assign(**dict(zip(cols,calibrated.T))),query.assign(**dict(zip(cols,future.T)))

def meta_predict(spec,train,query):
    cols = spec['columns']
    if spec.get('base_strength',0):
        train,query = calibrate_bases(train,query,cols,spec['base_strength'])
    x = train[cols].to_numpy(dtype=float)
    q = query[cols].to_numpy(dtype=float)
    y = train.y.to_numpy(dtype=float)
    kind = spec['kind']
    if kind=='weighted':
        weights = convex_weights(x,y)
        return x@weights,q@weights
    features = meta_context(train,cols)
    valid = meta_context(query,cols)
    if kind=='catboost':
        from catboost import CatBoostRegressor
        model = CatBoostRegressor(iterations=spec['iterations'],depth=spec['depth'],learning_rate=0.03,
                                  task_type='GPU',random_seed=42,thread_count=THREADS,allow_writing_files=False,verbose=False)
        base = x.mean(axis=1)
        model.fit(features,y-base)
        return np.maximum(base+model.predict(features),0),np.maximum(q.mean(axis=1)+model.predict(valid),0)
    import torch
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    scaler = StandardScaler().fit(features)
    def tensor(a):
        return torch.as_tensor(a,dtype=torch.float32,device='cuda')
    net = torch.nn.Sequential(torch.nn.Linear(features.shape[1],16),torch.nn.ReLU(),
                              torch.nn.Linear(16,len(cols)),torch.nn.Softmax(dim=1)).cuda()
    torch.nn.init.zeros_(net[-2].weight)
    torch.nn.init.zeros_(net[-2].bias)
    a,b = tensor(scaler.transform(features)),tensor(scaler.transform(valid))
    u,v,t = tensor(x),tensor(q),tensor(y)
    optimizer = torch.optim.Adam(net.parameters(),lr=0.001)
    for _ in range(spec['epochs']):
        optimizer.zero_grad()
        err = (net(a)*u).sum(dim=1)-t
        loss = (err.abs().sum()+err.sum().abs())/t.sum()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        return (net(a)*u).sum(dim=1).cpu().numpy(),(net(b)*v).sum(dim=1).cpu().numpy()

def specifications():
    for include in (False,True):
        cols = predictors(include)
        yield dict(kind='weighted',columns=cols)
        for depth in (3,5):
            for iterations in (100,300):
                yield dict(kind='catboost',depth=depth,iterations=iterations,columns=cols)
        for epochs in (50,150):
            yield dict(kind='gate',epochs=epochs,columns=cols)

def validation_rows(label,week,train,valid,pred_train,pred_valid):
    rows = []
    for split,frame,pred in [('train',train,pred_train),('validation',valid,pred_valid)]:
        for h in [0,*range(1,9)]:
            mask = np.ones(len(frame),dtype=bool) if h==0 else frame.horizon.eq(h)
            rows.append(dict(candidate=label,week=week,split=split,horizon=h,**metric(frame.y[mask],pred[mask])))
    display(pd.DataFrame(rows).query('horizon==0'))
    return rows

def compare_candidate(oof, spec, label, report, raw_label=None):
    rows = []
    past_actual,past_prediction = 0.,0.
    strength = spec.get('post_strength',0.)
    for week in (2,3,4):
        valid = oof.loc[oof.stage==f'oof{week}']
        train = oof.loc[oof.cutoff<valid.cutoff.min()]
        rel = f'meta/cv/{label}_{week}'
        if existing(rel+'.json'):
            cached = load_json(rel+'.json')
        else:
            tic = time.monotonic()
            if strength:
                raw = pd.read_parquet(require(f'meta/cv/{raw_label}_{week}.parquet')).prediction.to_numpy()
                factor = float(calibration_factors(past_actual,past_prediction,strength))
                p,q = raw[:len(train)]*factor,raw[len(train):]*factor
                raw_valid_volume = float(raw[len(train):].sum())
            else:
                p,q = meta_predict(spec,train,valid)
                raw_valid_volume = float(q.sum())
                if not spec.get('base_strength'):
                    save(rel+'.parquet',pd.DataFrame(dict(prediction=np.concatenate([p,q]))))
            model_rows = validation_rows(label,week,train,valid,p,q)
            baselines = []
            if spec['kind']=='weighted' and not (strength or spec.get('base_strength')):
                cols = spec['columns']
                single = min(cols,key=lambda c:metric(train.y,train[c])['score'])
                for key,tr,va in [(single,train[single],valid[single]),('mean',train[cols].mean(axis=1),valid[cols].mean(axis=1))]:
                    baselines.extend(validation_rows(f'baseline_{len(cols)}_{key}',week,train,valid,tr.to_numpy(),va.to_numpy()))
            cached = dict(model=model_rows,baselines=baselines,seconds=time.monotonic()-tic,
                          raw_valid_volume=raw_valid_volume,spec=spec)
            save(rel+'.json',cached)
        rows.extend(cached['model'])
        report.extend(cached['baselines'])
        past_actual += valid.y.sum()
        past_prediction += cached['raw_valid_volume']
    if strength:
        spec = dict(spec,post_factor=float(calibration_factors(past_actual,past_prediction,strength)),
                    post_oof_actual=float(past_actual),post_oof_prediction=float(past_prediction))
    valid_rows = [r for r in rows if r['split']=='validation' and r['horizon']==0]
    report.extend(rows)
    return dict(id=label,mean_week_score=float(np.mean([r['score'] for r in valid_rows])),
                pooled_score=combine(valid_rows)['score'],kind=spec['kind'],n_columns=len(spec['columns']),spec=spec)

def report_base_calibration(oof):
    rows = []
    for week in (2,3,4):
        valid = oof.loc[oof.stage==f'oof{week}']
        past = oof.loc[oof.cutoff<valid.cutoff.min()]
        cols = predictors(True)
        for strength in (0.,0.5,1.):
            factors = calibration_factors(past.y.sum(),past[cols].sum().to_numpy(),strength)
            for col,factor in zip(cols,factors):
                rows.append(dict(week=week,model=col,strength=strength,factor=factor,
                                 **metric(valid.y,valid[col]*factor)))
    table = pd.DataFrame(rows)
    save('meta/base_calibration.parquet',table)
    display(table.pivot_table(index='model',columns='strength',values='score'))

def assemble():
    oof = pd.concat([collect_stage(f'oof{i}',predictors(True)) for i in range(1,5)],ignore_index=True)
    if len(oof)!=224000 or oof.duplicated(KEY).any():
        raise ValueError('OOF должен содержать 224000 уникальных строк')
    save('meta/oof.parquet',oof)
    report_base_calibration(oof)
    report = []
    leaderboard = [compare_candidate(oof,spec,f'raw_{i:02}',report) for i,spec in enumerate(specifications())]
    def rank(entry):
        spec = entry['spec']
        return (entry['mean_week_score'],entry['n_columns'],entry['kind']!='weighted',
                bool(spec.get('base_strength') or spec.get('post_strength')),entry['id'])
    best = min(leaderboard,key=rank)
    for mode in ('base_strength','post_strength'):
        for strength in (0.5,1.0):
            spec = dict(best['spec'],**{mode:strength})
            leaderboard.append(compare_candidate(oof,spec,f'{mode}_{strength}',report,best['id']))
    ordered = sorted(leaderboard,key=rank)
    save('meta/validation.parquet',pd.DataFrame(report))
    save('meta/leaderboard.json',ordered)
    save('meta/winner.json',ordered[0]['spec'])
    display(pd.DataFrame(ordered).drop(columns='spec'))

def save_calibration(stage,spec,train,post_factor):
    cols = spec['columns']
    factors = calibration_factors(train.y.sum(),train[cols].sum().to_numpy(),spec.get('base_strength',0.))
    save(f'meta/{stage}_calibration.parquet',pd.DataFrame(dict(model=[*cols,'ensemble'],
                                                              factor=[*factors,post_factor])))

def evaluate():
    winner = load_json('meta/winner.json')
    oof = pd.read_parquet(require('meta/oof.parquet'))
    test = collect_stage('test',winner['columns'])
    a,b = meta_predict(winner,oof,test)
    raw_test = b.copy()
    post_factor = winner.get('post_factor',1.)
    a,b = a*post_factor,b*post_factor
    save_calibration('test',winner,oof,post_factor)
    rows = {'train':metric(oof.y,a),'test':metric(test.y,b)}
    save('meta/test_predictions.parquet',test.assign(prediction=b,raw_prediction=raw_test))
    save('meta/test_report.json',dict(winner=winner,metrics=rows,note='Независимая оценка test получена до переобучения на всей истории. Этот период ранее использовался в проекте.'))
    display(pd.DataFrame(rows).T)

def submit():
    winner = load_json('meta/winner.json')
    if load_json('meta/test_report.json')['winner']!=winner:
        raise ValueError('Изменилась конфигурация после оценки')
    cols = winner['columns']
    test = pd.read_parquet(require('meta/test_predictions.parquet'))
    oof = pd.read_parquet(require('meta/oof.parquet'))
    train = pd.concat([oof,test.loc[test.cutoff<TEST_START+7*DAY],collect_stage('extra',cols)],ignore_index=True)
    if len(train)!=336000 or train.duplicated(KEY).any():
        raise ValueError('Неполные финальные OOF')
    forecast = collect_stage('full',cols)
    a,p = meta_predict(winner,train,forecast)
    post_factor = 1.
    if winner.get('post_strength'):
        post_factor = float(calibration_factors(winner['post_oof_actual']+test.y.sum(),
                                                winner['post_oof_prediction']+test.raw_prediction.sum(),
                                                winner['post_strength']))
    a,p = a*post_factor,p*post_factor
    save_calibration('full',winner,train,post_factor)
    display(pd.DataFrame([metric(train.y,a)],index=['full OOF train']))
    forecast['y_pred'] = p
    result = pd.read_parquet(TEST_PATH)[['id','route_id','timestamp']].merge(
        forecast[['route_id','timestamp','y_pred']],on=['route_id','timestamp'],how='left',validate='one_to_one',sort=False)
    if len(result)!=8000 or not result.id.is_unique or not np.isfinite(result.y_pred).all() or (result.y_pred<0).any():
        raise ValueError('Нарушен контракт submission')
    path = ROOT/('submission_fast.csv' if PROFILE=='fast' else 'submission.csv')
    temp = path.with_suffix('.partial')
    result[['id','y_pred']].to_csv(temp,index=False)
    temp.replace(path)
    print(path)
''')

In [ ]:
ENGINE.append(r'''
if MODE not in ('oof','assemble','evaluate','submit') or ROLE not in ('worker','collect'):
    raise ValueError('Проверьте MODE и ROLE')
if MODE=='assemble' or ROLE=='collect':
    if PART!=1 or MODE=='oof':
        raise ValueError('Сборку выполняет первая часть в assemble/evaluate/submit')
    {'assemble':assemble,'evaluate':evaluate,'submit':submit}[MODE]()
else:
    run_worker()
''')

In [ ]:
import json
import os
import signal
import subprocess
import sys
import tempfile
from pathlib import Path
from threading import Thread
from time import monotonic

if not 0 < SESSION_HOURS <= 11.5:
    raise ValueError('SESSION_HOURS должен быть в диапазоне (0, 11.5]')
configuration = ('PART','HORIZONS','MODE','ROLE','OOF_WEEKS','INPUT_DIR','OUTPUT_DIR',
                 'CODE_ID','PARAMS','PROFILE','ROUTE_ALPHA')
setup = '\n'.join(f'{name} = {globals()[name]!r}' for name in configuration)
scratch = Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else None
with tempfile.TemporaryDirectory(dir=scratch) as folder:
    script = Path(folder)/'run_stack.py'
    script.write_text('\n\n'.join([setup,*ENGINE]))
    process = subprocess.Popen([sys.executable,'-u',str(script)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                               env=dict(os.environ,TMPDIR=folder),text=True,errors='replace',start_new_session=True)
    def show_output():
        for line in process.stdout:
            print(line,end='',flush=True)
    reader = Thread(target=show_output,daemon=True)
    reader.start()
    try:
        process.wait(timeout=max(0.,SESSION_HOURS*3600-(monotonic()-SESSION_START)))
    except subprocess.TimeoutExpired:
        if process.poll() is None:
            os.killpg(process.pid,signal.SIGKILL)
        process.wait()
        status = 'time_limit'
        print('Бюджет сессии исчерпан. Подключите сохранённый output и предыдущие части, затем продолжите тот же этап.')
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid,signal.SIGKILL)
        process.wait()
        raise
    else:
        status = 'complete'
        if process.returncode:
            raise RuntimeError(f'Этап завершился с кодом {process.returncode}. См. вывод выше.')
    finally:
        reader.join()
    status_path = Path(OUTPUT_DIR)/f'session_part{PART}_{PROFILE}_{MODE}_{ROLE}.json'
    status_path.parent.mkdir(parents=True,exist_ok=True)
    status_path.write_text(json.dumps(dict(status=status,elapsed_hours=(monotonic()-SESSION_START)/3600)))